In [19]:
pip install pandas scikit-learn emlearn numpy

In [21]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import emlearn

# 1. Setup portabili con percorsi relativi
target_directory = "dataset"
os.makedirs(target_directory, exist_ok=True)

csv_output_path = os.path.join(target_directory, "smart_health_dataset.csv")
header_output_path = os.path.join(target_directory, "vitals_classifier.h")

# 2. GENERATION OF THE 1500 CLINICAL SAMPLES
print("Generating 1500 clinical samples for the dataset...")
np.random.seed(42)
total_samples = 1500
clinical_data = []

for _ in range(total_samples):
    # 70% chance of healthy patient (Risk=0), 30% chance of emergency (Risk=1)
    is_emergency = np.random.choice([0, 1], p=[0.7, 0.3])
    
    if is_emergency == 0:
        # Healthy Patient: Normal Heart Rate (60-100 bpm) and stable SpO2 (95-100%)
        heart_rate = np.random.randint(60, 101)
        spo2 = np.random.randint(95, 101)
        risk_label = 0
    else:
        # Emergency Patient: Randomly select a critical clinical scenario
        medical_scenario = np.random.choice(['tachycardia', 'bradycardia', 'hypoxia'])
        
        if medical_scenario == 'tachycardia':
            # Extreme high heart rate, possible panic or cardiac issue
            heart_rate = np.random.randint(120, 185)
            spo2 = np.random.randint(92, 99)
        elif medical_scenario == 'bradycardia':
            # Dangerous low heart rate, bradycardia
            heart_rate = np.random.randint(30, 50)
            spo2 = np.random.randint(92, 99)
        else: # hypoxia
            # Critical low blood oxygen level (Hypoxia)
            heart_rate = np.random.randint(65, 110)
            spo2 = np.random.randint(75, 92)
        risk_label = 1
        
    clinical_data.append([heart_rate, spo2, risk_label])

# 3. EXPORT TO CSV (Using ';' separator just like the occupancy project)
dataframe = pd.DataFrame(clinical_data, columns=['HeartRate', 'SpO2', 'Risk'])
dataframe.to_csv(csv_output_path, sep=';', index=False)
print(f"[✓] File successfully created with {len(dataframe)} rows at: {csv_output_path}")

# 4. TRAINING THE RANDOM FOREST MODEL
print("Training the Random Forest model on the generated dataset...")
X_train = dataframe[['HeartRate', 'SpO2']].values
y_train = dataframe['Risk'].values

# Light configuration optimized for Contiki-NG constraints (LAB 02 Slide 18)
random_forest_model = RandomForestClassifier(n_estimators=5, max_depth=3, random_state=42)
random_forest_model.fit(X_train, y_train)
print("[✓] Model training completed successfully.")

# 5. EXPORT TO EMBEDDED C HEADER USING EMLEARN
print("Converting model into C code via emlearn...")
c_model = emlearn.convert(random_forest_model, method='inline')
c_model.save(file=header_output_path, name='vitals_rf_model')
print(f"[✓] C Header file generated at: {header_output_path}")
print("\n[SUCCESS] Execution finished. Both files are local on your machine!")

Generating 1500 clinical samples for the dataset...
[✓] File successfully created with 1500 rows at: dataset/smart_health_dataset.csv
Training the Random Forest model on the generated dataset...
[✓] Model training completed successfully.
Converting model into C code via emlearn...
[✓] C Header file generated at: dataset/vitals_classifier.h

[SUCCESS] Execution finished. Both files are local on your machine!
